# Lab 10: Data Analysis (Music Analytics)
This notebook explores the cleaned music dataset and fulfills all assignment criteria.

In [ ]:
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath('../src'))
import analytics.db_connector as db
import analytics.data_combiner as dc
import analytics.pivot_builder as pb
import analytics.aggregator as ag
import analytics.time_series as ts
import analytics.mongo_pipeline as mp
import analytics.insight_reporter as ir
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/cleaned/clean.csv')

## Part 1: MySQL Connection and Population

In [ ]:
try:
    conn = db.get_connection()
    db.populate_financials(df, conn)
    sql_df = db.query_financials(conn)
    print("Successfully queried MySQL. Rows:", len(sql_df))
    conn.close()
except Exception as e:
    print("MySQL not running or credentials invalid. Skipping DB step.")

## Part 2: Combining Data Sources

In [ ]:
df_part1 = df[['name', 'listeners']].head(50)
df_part2 = df[['name', 'playcount']].tail(50)

for join_type in ['inner', 'left', 'right', 'outer']:
    joined = dc.combine_data(df_part1, df_part2, on_key='name', how=join_type)
    print(f"{join_type} join row count: {len(joined)}")

## Part 3: Data Reshaping

In [ ]:
long_df = pb.convert_wide_to_long(df, ['name', 'source_collection'], ['listeners', 'playcount'])
print("Wide to Long format:")
print(long_df.head())

## Part 4: GroupBy Analysis (Aggregations and Top N)

In [ ]:
summary = df.groupby('source_collection').agg(
    avg_listeners=('listeners', 'mean'),
    total_listeners=('listeners', 'sum'),
    count_artists=('name', 'count'),
    median_playcount=('playcount', 'median')
)
print(summary)
summary.to_csv('../data/processed/analytics/genre_analysis.csv')

top_n = ag.top_n_per_group(df, group_col='source_collection', sort_col='listeners', n=3)
print("\nTop 3 per group:\n", top_n[['source_collection', 'name', 'listeners']])

## Part 5: Pivot Tables

In [ ]:
df = ts.parse_dates(df, 'release_date')
pivot = pb.build_pivot_table(df, 'year', 'source_collection', 'listeners', 'sum')
print("Pivot Table (margins=True):")
print(pivot.tail())
pivot.to_csv('../data/processed/analytics/pivot_genre_year.csv')

## Part 6: Time Series (Rolling Averages)

In [ ]:
monthly = ts.resample_data(df, 'release_date', 'listeners', rule='ME')
monthly['rolling_3M'] = ts.calculate_rolling_avg(monthly, 'listeners', 3)
monthly['rolling_6M'] = ts.calculate_rolling_avg(monthly, 'listeners', 6)
monthly['rolling_12M'] = ts.calculate_rolling_avg(monthly, 'listeners', 12)
print("Monthly Resampling with Rolling Averages:")
print(monthly.tail())

## Part 7: MongoDB Aggregation Pipeline

In [ ]:
print("MongoDB Pipeline constructed:")
print(mp.build_aggregation_pipeline())

## Part 8: Analytical Questions

In [ ]:
ir.run_all_questions(df)
ir.plot_genre_roi(df.head(20), output_path='../data/processed/analytics/genre_roi.png')